# Train YOLO `best.pt` for Store Intelligence

Run this notebook in Google Colab with GPU enabled. It trains a YOLO person detector and saves the final model as `best.pt`, which you should place in the project directory at `models/best.pt`.

Important: raw CCTV videos are not enough for supervised training. You need labeled person bounding boxes in YOLO format, or a Roboflow/CVAT/Label Studio export that includes `data.yaml`.

In [ ]:
!nvidia-smi
!pip install -q ultralytics roboflow opencv-python-headless pyyaml

from google.colab import drive, files
drive.mount('/content/drive')

## Option A: Use a YOLO-format dataset from Google Drive

Expected structure:

```text
/content/drive/MyDrive/store-intelligence-yolo/
  data.yaml
  train/images/*.jpg
  train/labels/*.txt
  valid/images/*.jpg
  valid/labels/*.txt
```

The `data.yaml` should contain one class for person detection, for example:

```yaml
path: /content/drive/MyDrive/store-intelligence-yolo
train: train/images
val: valid/images
names:
  0: person
```

In [ ]:
from pathlib import Path
import shutil
import yaml

DATASET_YAML = Path('/content/drive/MyDrive/store-intelligence-yolo/data.yaml')
OUTPUT_DIR = Path('/content/drive/MyDrive/store-intelligence-models')
BASE_MODEL = 'yolov8n.pt'  # use yolov8s.pt for better accuracy if GPU memory allows
EPOCHS = 60
IMG_SIZE = 640
BATCH = 16

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Dataset YAML:', DATASET_YAML)
print('Output dir:', OUTPUT_DIR)
assert DATASET_YAML.exists(), 'Upload/export a YOLO dataset and update DATASET_YAML.'

with DATASET_YAML.open('r') as f:
    data_config = yaml.safe_load(f)
print(data_config)

## Annotation protocol for this project

Use a single YOLO object-detection class:

```text
person
```

Label every visible human as `person`, including customers and staff. Do not create separate YOLO classes for `staff`, `customer`, `queue`, `entry`, or store zones in the first detector. The API/pipeline handles staff flags, queue depth, zone mapping, entry/exit, and re-entry after person detection.

Bounding-box rules:

- Draw one tight box per visible person, not one box around a group.
- Include the visible head/torso/body, even when faces are blurred.
- For partial occlusion, label the visible body region if roughly 30% or more is visible.
- Skip tiny fragments such as only hands/feet, posters, mannequins, reflections, and product images.
- Keep empty-store frames with zero labels so the model learns no-person scenes.

Dataset coverage checklist:

- Entry/exit threshold frames.
- Group entry with 2-4 people.
- Staff moving through zones.
- Crowded billing queue frames.
- Empty store periods.
- Partial occlusion behind displays or other customers.
- All camera angles and lighting conditions.

Recommended split: 70% train, 20% validation, 10% test. Prefer splitting by time segment/camera clip rather than purely random consecutive frames because CCTV frames are highly similar.

## Roboflow annotation and export workflow

1. Create a Roboflow **Object Detection** project.
2. Add exactly one class: `person`.
3. Upload the frames you already extracted.
4. Annotate using the protocol above.
5. Keep some empty frames with zero boxes.
6. Generate a dataset version.
7. Use preprocessing: auto-orient enabled, resize to 640x640, letterbox/fit if available.
8. Use only light CCTV-realistic augmentations: brightness/exposure, small blur/noise, optional horizontal flip.
9. Avoid vertical flip, extreme rotation, and heavy crops.
10. Export/download as **YOLOv8** format, or use the Roboflow code block below.

## Option B: Download a labeled dataset from Roboflow

Use this only if your annotated challenge frames are in Roboflow. Set `USE_ROBOFLOW = True` and fill the workspace/project/version values. The code updates `DATASET_YAML` automatically.

In [ ]:
USE_ROBOFLOW = False

if USE_ROBOFLOW:
    from getpass import getpass
    from roboflow import Roboflow

    ROBOFLOW_API_KEY = getpass('Roboflow API key: ')
    ROBOFLOW_WORKSPACE = 'your-workspace'
    ROBOFLOW_PROJECT = 'your-project'
    ROBOFLOW_VERSION = 1

    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
    dataset = project.version(ROBOFLOW_VERSION).download('yolov8')
    DATASET_YAML = Path(dataset.location) / 'data.yaml'
    print('Downloaded dataset:', DATASET_YAML)

    with DATASET_YAML.open('r') as f:
        data_config = yaml.safe_load(f)
    print('Roboflow data.yaml:', data_config)
    names = data_config.get('names', {})
    print('Class names:', names)
    assert 'person' in list(names.values()) or names == ['person'], 'Expected a single person class for this detector.'

## Optional: Extract frames from raw CCTV clips for labeling

This does not train a model by itself. It creates images that you can upload into CVAT, Roboflow, or Label Studio for person-box annotation.

In [ ]:
EXTRACT_FRAMES = False
VIDEO_DIR = Path('/content/drive/MyDrive/store-intelligence-videos')
FRAME_OUTPUT = Path('/content/drive/MyDrive/store-intelligence-frames')
SECONDS_BETWEEN_FRAMES = 2

if EXTRACT_FRAMES:
    import cv2
    FRAME_OUTPUT.mkdir(parents=True, exist_ok=True)
    for video_path in sorted(VIDEO_DIR.glob('*.mp4')):
        cap = cv2.VideoCapture(str(video_path))
        fps = cap.get(cv2.CAP_PROP_FPS) or 15
        step = max(1, int(fps * SECONDS_BETWEEN_FRAMES))
        frame_index = 0
        saved = 0
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            if frame_index % step == 0:
                out = FRAME_OUTPUT / f'{video_path.stem}_{frame_index:06d}.jpg'
                cv2.imwrite(str(out), frame)
                saved += 1
            frame_index += 1
        cap.release()
        print(video_path.name, 'frames saved:', saved)
    print('Upload these frames to a labeling tool, export YOLO format, then run training.')

## Train the YOLO model

In [ ]:
from ultralytics import YOLO

model = YOLO(BASE_MODEL)
results = model.train(
    data=str(DATASET_YAML),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    project=str(OUTPUT_DIR / 'runs'),
    name='person_detector',
    exist_ok=True,
    patience=15,
    workers=2,
    device=0,
)

best_pt = Path(results.save_dir) / 'weights' / 'best.pt'
print('Best checkpoint:', best_pt)
assert best_pt.exists(), 'Training finished but best.pt was not found.'

## Validate, save, and download `best.pt`

In [ ]:
trained = YOLO(str(best_pt))
metrics = trained.val(data=str(DATASET_YAML), split='val')
print(metrics)

final_model = OUTPUT_DIR / 'best.pt'
shutil.copy2(best_pt, final_model)
print('Saved final model to:', final_model)

files.download(str(final_model))

## Optional: Smoke-test on one CCTV clip

In [ ]:
TEST_VIDEO = Path('/content/drive/MyDrive/store-intelligence-videos/CAM 1.mp4')

if TEST_VIDEO.exists():
    trained = YOLO(str(final_model))
    predictions = trained.predict(
        source=str(TEST_VIDEO),
        conf=0.25,
        vid_stride=5,
        save=True,
        project=str(OUTPUT_DIR / 'predictions'),
        name='cctv_smoke_test',
        exist_ok=True,
    )
    print('Prediction output saved under:', OUTPUT_DIR / 'predictions')
else:
    print('Skipping smoke test because TEST_VIDEO does not exist:', TEST_VIDEO)

## Use the model in the project

After downloading `best.pt`, place it in the project folder:

```text
models/best.pt
```

Then run:

```bash
pip install -r requirements-pipeline.txt
python -m pipeline.run --mode detect --video "CCTV Footage/CAM 1.mp4" --camera-id CAM_1 --model models/best.pt --output generated_events.jsonl
```